In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import sklearn
import random
import tensorflow_hub as hub
import keras_nlp

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

# **Estudio y análisis de técnicas de few-shot machine learning**

## Introducción

El **few-shot learning** es un marco del machine learning en el que un modelo de IA aprende a hacer predicciones precisas pese a contar un número muy pequeño de ejemplos etiquetados. Por lo general, se usa para entrenar modelos para tareas de clasificación cuando los datos de entrenamiento adecuados son escasos. Se suele considerar un subproblema del **n-way k-shot learning**, una rama de estudio en la que se busca resolver un problema de clasificación con **n** clases contando con **k** ejemplos etiquetados por cada clase, se encuentran en esta rama de estudio n también el **one-shot learning** donde se cuenta ya con 1 ejemplo por clase o el **zero-shot learning** donde no se tienen ejemplos en absoluto.









Se considera que un dataset cuenta con pocos ejemplos etiquetados, y que por tanto se trata de un problema de few-shot learning cuando dicho dataset cuenta con menos de 10 ejemplos por clase. Es por ello que el few shot learning se plantea principalmente en problemas de clasificación multiclase.



Existe también el término de **few-label learning**, que refiere a problemas más específicos en los que en el dataset cuenta con una mayor cantidad de ejeplos con el condicionante de que estos no se encuentran etiquetados.

## Carga de Datos

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
SEED=42
def create_few_shot_splits(dataset_name="banking77", k=5, seed=SEED):
    """
    Carga el dataset y crea splits few-shot.
    datset_name: conjunto de datos de la librería 'datasets' que se usará.
    k: número de ejemplos por clase (shots).
    seed: semilla para reproducibilidad.
    """
    print(f"--- Cargando {dataset_name} ---")
    dataset = load_dataset(dataset_name)
    
    df_train = pd.DataFrame(dataset['train'])
    df_train, df_val = train_test_split(
        df_train, 
        test_size=0.20, 
        random_state=seed, 
        stratify=df_train['label']
    )
    df_test = pd.DataFrame(dataset['test'])

    labels = df_train['label'].unique()

    few_shot_data = []
    unlabeled_data = []

    for label in labels:
        df_class = df_train[df_train['label'] == label]

        if len(df_class) < k:
            sampled = df_class
            few_shot_data.append(sampled)
        else:
            sampled = df_class.sample(n=k, random_state=seed)
            remaining = df_class.drop(sampled.index)
            few_shot_data.append(sampled)
            unlabeled_data.append(remaining)

    df_few_shot = pd.concat(few_shot_data).sample(frac=1, random_state=seed).reset_index(drop=True)
    df_unlabeled = pd.concat(unlabeled_data).sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"Resumen de Splits (Seed={SEED}):")
    print(f"1. Few-Shot Train ({k}-shot): {len(df_few_shot)}")
    print(f"2. Unlabeled Train: {len(df_unlabeled)}")
    print(f"3. Validation Set: {len(df_val)}")
    print(f"4. Test Set: {len(df_test)} ")

    return df_few_shot,  df_unlabeled,df_val, df_test


In [ ]:
K=10
train, _,_, test = create_few_shot_splits(dataset_name="SetFit/20_newsgroups", k=10)
print("\nEjemplo de datos:")
print(train[['text', 'label']].head())

In [ ]:
K=10
train_fs, train_ul, val, test = create_few_shot_splits(dataset_name="SetFit/20_newsgroups", k=K)
df_ordenado = train_fs.copy()
df_ordenado['order_id'] = df_ordenado.groupby('label').cumcount()

train_fs_list = []

for i in range(1, 11):
    k_shot_train = df_ordenado[df_ordenado['order_id'] < i].drop(columns=['order_id'])
    k_shot_train = k_shot_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
    train_fs_list.append(k_shot_train)


A continuación se segmenta este dataset en los conjuntos desde 1 shot hasta 10 shot que se usarán en las pruebas.

In [ ]:
for idx, k_shot in enumerate(train_fs_list):
    ejemplos_por_clase = k_shot['label'].value_counts().iloc[0]
    print(f"Conjunto {idx + 1}: Total filas = {len(k_shot)} ({ejemplos_por_clase} ejemplos por clase)")
train_fs_list[1]

## Vectorización del texto

In [ ]:
import keras_nlp

TOKENIZER = keras_nlp.models.BertTokenizer.from_preset(
    "bert_base_multi", 
    sequence_length=256
)

tokens = TOKENIZER(train['text'].values)

print("Tokens (IDs numéricos):")
print(tokens)

print("\nPrimer texto decodificado por fragmentos:")
print(TOKENIZER.detokenize(tokens[0])) 

## Red Neuronal base

In [ ]:
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.callbacks import EarlyStopping

def create_fs_NN(X, y, X_val=np.array([]), y_val=np.array([]),num_layers = 1, layers_size=128, activation='relu', early_stopping=False,embedding=True, learning_rate=0.01, epochs =30, batch_size= 10, train=True,verbose=2):
    num_classes = np.unique(y).size
    model = models.Sequential()
    if embedding:
        X = tf.constant(X, dtype=tf.string)
        y = tf.constant(y, dtype=tf.int32)
        X_val = tf.constant(X_val, dtype=tf.string)
        y_val = tf.constant(y_val, dtype=tf.int32)
        max_tokens = 1000
        vectorizer = TextVectorization(max_tokens=max_tokens, output_mode='int', output_sequence_length=256)
        vectorizer.adapt(X)
        model.add(vectorizer)
        model.add(layers.Embedding(max_tokens, layers_size))
        model.add(layers.GlobalAveragePooling1D())

    else:
        X = tf.constant(X, dtype=tf.int32)
        y = tf.constant(y, dtype=tf.int32)
        X_val = tf.constant(X_val, dtype=tf.int32)
        y_val = tf.constant(y_val, dtype=tf.int32)


    for i in range(num_layers):
        model.add(layers.Dense(layers_size, activation=activation))


    model.add(layers.Dense(num_classes, activation='softmax'))



    model.compile(optimizer= optimizers.Adam(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'],jit_compile=False)

    if train:
        
        if early_stopping:
        
            early_stopping = EarlyStopping(
                monitor='val_loss',       
                patience=5,              
                restore_best_weights=True 
            )

        
            history = model.fit(X, y,
                            validation_data=(X_val, y_val),
                            epochs=epochs,
                            batch_size=batch_size,
                            callbacks=[early_stopping],    
                            verbose=verbose)
            
        else:
            
            history = model.fit(X, y,
                  epochs=epochs,
                  batch_size=batch_size,
                  verbose= verbose)
    return model

A continuación se muestran las características del modelo para el dataset 10-shot de banking77, con los hiperparámetros establecidos a los valores por defecto.

In [ ]:
train = train_fs_list[9]

#X = TOKENIZER(train['text'].values)
X = train['text'].values
y = train['label'].values

model = create_fs_NN(X,y, verbose=0)
model.summary()

### Optimización de hiperparámetros

In [ ]:
NN_param_values = {'num_layers': [1,2,3], 
                'layers_size' : [128,256,512], 
                'activation' : ['relu'],
                'embedding' :[True,False],
                'early_stopping': [True,False],
                'learning_rate' : [0.01, 1e-3, 1e-4],
                'epochs' : [50],
                'batch_size' : [5,10,20,40,60],
                'train' : [True]
            }

In [ ]:
from sklearn.model_selection import train_test_split
import itertools
def Grid_search(X_train, y_train, X_val, y_val, model_function, params):

    mejor_score = - np.inf
    
    best_params = dict()
    keys = list(params.keys())
    values = list(params.values())
    num_updates = 0
    for combination in itertools.product(*values):
        params_it = dict(zip(keys, combination))
        if params_it['embedding']:
            model_it = model_function(X_train, y_train, X_val, y_val,**params_it, verbose=1)
            y_pred_proba = model_it.predict(X_val)

        else:
            X_train_token = TOKENIZER(X_train)
            X_val_token = TOKENIZER(X_val)
            model_it = model_function(X_train_token, y_train, X_val_token, y_val, **params_it, verbose=1)
            y_pred_proba = model_it.predict(X_val_token)
        
        y_pred = np.argmax(y_pred_proba, axis=1)
        it_score = sum(y_pred == y_val) / len(y_val)

        if it_score >= mejor_score:
            num_updates += 1
            print("Actualizacion". num_updates)
            print("Nuevo mejor:", params_it, it_score)
            mejor_score = it_score 
            mejor = model_it
            best_params = params_it
    return best_params

In [ ]:
train = train_fs_list[9]

X_train = train['text'].values
y_train = train['label'].values
X_val = val['text'].values
y_val = val['label'].values
#best_params_NN = Grid_search(X_train,y_train, X_val, y_val,create_fs_NN,NN_param_values)

In [ ]:
best_params_NN = {'num_layers': 1, 'layers_size': 512, 'activation': 'relu', 'embedding': True, 'early_stopping': False, 'learning_rate': 0.001, 'epochs': 50, 'batch_size': 10, 'train': True}


In [ ]:
print(best_params_NN)

### Evaluación del modelo

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

def kscores(function, k_values, params=dict(),embedding=True):
    micro_f1_scores = []
    macro_f1_scores = []

    for k in k_values:
        train = train_fs_list[k]

        if embedding:
            
            X = train['text'].values
            X_val = val['text'].values
            X_test = test['text'].values
            X_test = tf.constant(X_test, dtype=tf.string)
            
        else:
            X = TOKENIZER( train['text'].values)
            X_val = TOKENIZER(val['text'].values)
            X_test = TOKENIZER(test['text'].values)
            X_test = tf.constant(X_test, dtype=tf.int32)


        y = train['label'].values
        y_val = val['label'].values
        y_test = test['label'].values
        y_test = tf.constant(y_test, dtype=tf.int32)

        model = function(X,y,X_val,y_val,**params,verbose=1)
        y_pred_proba = model.predict(X_test)
        y_pred = np.argmax(y_pred_proba, axis=1)

        micro_f1 = f1_score(y_test, y_pred, average='micro')
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        micro_f1_scores.append(micro_f1)
        macro_f1_scores.append(macro_f1)

    return micro_f1_scores, macro_f1_scores




In [ ]:

#micro_f1_scores_nn, macro_f1_scores_nn = kscores(create_fs_NN, range(1,10), best_params_NN, embedding=best_params_NN['embedding'])

In [ ]:
best_params_vanilla = best_params_NN.copy()
best_params_vanilla['early_stopping'] = True
#micro_f1_scores_nn_vanilla, macro_f1_scores_nn_vanilla = kscores(create_fs_NN, range(1,10), best_params_vanilla, embedding=best_params_vanilla['embedding'])

In [ ]:
best_params_vec = best_params_NN.copy()
best_params_vec['embedding'] = False
#micro_f1_scores_nn_vec, macro_f1_scores_nn_vec = kscores(create_fs_NN, range(1,10), best_params_vec, embedding=False)

In [ ]:
#Resultados del test
micro_f1_scores_nn, macro_f1_scores_nn = [0.0768720127456187, 0.08032395114179501, 0.096388741370154, 0.10023898035050452, 0.10939989378651088, 0.13528943175783326, 0.1421933085501859, 0.15029208709506106, 0.18003186404673394] , [0.05595976119115738, 0.07123757293363504, 0.08591559144954151, 0.0871167970260494, 0.10640314497776308, 0.12598091798968472, 0.13450868393060283, 0.14100967167147435, 0.16580862270514235]
micro_f1_scores_nn_vanilla, macro_f1_scores_nn_vanilla = [0.05483271375464684, 0.05377057886351567, 0.0723579394583112, 0.06386086032926182, 0.05881571959638874, 0.06890600106213489, 0.0686404673393521, 0.06293149229952204, 0.062267657992565055] , [0.017373176602119306, 0.009831649831649832, 0.024542223920605618, 0.01693990541542344, 0.017205990473380377, 0.02289430664858172, 0.025445166382758937, 0.01662028585969074, 0.014509476632234054]
micro_f1_scores_nn_vec, macro_f1_scores_nn_vec = [0.04221986192246415, 0.049787573021773765, 0.049522039298990975, 0.058417419012214554, 0.05363781200212427, 0.06067445565586829, 0.04792883696229421, 0.04753053637812002, 0.05562931492299522] , [0.026597812011502537, 0.03829683967733839, 0.04247600489148001, 0.04966411745621908, 0.04971935651210194, 0.05204901612023941, 0.04405917375987634, 0.03960173345582641, 0.04846134805823056]

In [ ]:
#Imprimimos los resultados del test
print(micro_f1_scores_nn, ",", macro_f1_scores_nn )

In [ ]:
#Imprimimos los resultados del test (sin early stopping)
print(micro_f1_scores_nn_vanilla, ",", macro_f1_scores_nn_vanilla )

In [ ]:
#Imprimimos los resultados del test (sin incluir capa de embedding)
print(micro_f1_scores_nn_vec, ",", macro_f1_scores_nn_vec )

A continuación se muestras los resultados de las puntuaciones:

In [ ]:
plt.figure(figsize=(15, 5))

plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)

plt.plot(range(2,11), micro_f1_scores_nn, marker="o", label="Sin Early Stopping")
plt.plot(range(2,11), micro_f1_scores_nn_vanilla, marker="o", label="Con Early Stopping")
plt.plot(range(2,11), micro_f1_scores_nn_vec, marker="o", label="Sin incluir capa de Embedding")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(range(2,11), macro_f1_scores_nn, marker="o", label="Sin Early Stopping")
plt.plot(range(2,11), macro_f1_scores_nn_vanilla, marker="o", label="Con Early Stopping")
plt.plot(range(2,11), macro_f1_scores_nn_vec, marker="o", label="Sin incluir capa de Embedding")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()
plt.grid(True)
plt.savefig('grafica1.png', bbox_inches='tight')
plt.show()


## Data Augmentation

In [ ]:
import nlpaug.augmenter.word as naw

#Puede dar problemas con los drivers, incluso al ejcutar en CPU
#aug_bert = naw.ContextualWordEmbsAug(model_path='distilbert-base-uncased', action="substitute", device='cpu')

aug_syn = naw.SynonymAug(aug_src='wordnet')

aug_swap = naw.RandomWordAug(action="swap", aug_p=0.1)

def apply_custom_augmentation(X,y, bert_aug=5, syn_aug=3, swap_aug=1):
    new_data = []
    print('Aumentando los datos del dataset')
    for i in range(X.shape[0]):

        text_original = X[i]
        label = y[i]

        #text_sem = aug_bert.augment(text_original)[:bert_aug]
        text_syn = aug_syn.augment(text_original)[:syn_aug]
        text_swap = aug_swap.augment(text_original)[:swap_aug]

        new_data.append({'text': text_original, 'label': label})
        for text in text_sem:
            new_data.append({'text': text, 'label': label})
        for text in text_syn:
            new_data.append({'text': text, 'label': label})
        for text in text_swap:
            new_data.append({'text': text, 'label': label})

    df_aug = pd.DataFrame(new_data)
    df_desordenado = df_aug.sample(frac=1).reset_index(drop=True)
    return df_desordenado['text'].values, df_desordenado['label'].values




Si componenmos la función con la  función del anteior modelo tenemos un modelo que incorpora Data Augmentation para resolver el problema.

In [ ]:
create_fs_nn_aug = lambda X, y, X_val=np.array([]), y_val=np.array([]),bert_aug=1, syn_aug=1, swap_aug=1, nn_params=best_params_NN, verbose=1, embedding=True : create_fs_NN(*apply_custom_augmentation(X, y,bert_aug, syn_aug, swap_aug), X_val=X_val, y_val = y_val,verbose=verbose,**nn_params)

### Optimización de Hiperparámetros

In [ ]:
best_params_aug = {'bert_aug': 5, 'syn_aug': 5, 'swap_aug': 5, 'nn_params': best_params_NN}


In [ ]:
print(best_params_aug)

### Evaluación del modelo



Empleando la función **k_scores** obtenemos las puntuaciones del modelo:

In [ ]:
k_values = range(1,10)

#micro_f1_scores_nn_aug, macro_f1_scores_nn_aug = kscores(create_fs_nn_aug, k_values,best_params_aug)

In [ ]:
#Resultados obtenidos en el último test
micro_f1_scores_nn_aug = [0.06346255974508763, 0.09426447158789167, 0.0845724907063197, 0.09227296866702071, 0.1165693043016463, 0.1253319171534785, 0.13528943175783326, 0.13967073818374934, 0.14352097716409984]
macro_f1_scores_nn_aug = [0.050019384132770794, 0.07777958088769338, 0.07217174427451356, 0.08219883014830852, 0.11119322812385635, 0.11670819561905081, 0.13016933317741805, 0.13776498078442798, 0.13972103109715409]


In [ ]:
#Mostramos los resultados para guardarlos en el notebook
print(micro_f1_scores_nn_aug)
print(macro_f1_scores_nn_aug)

In [ ]:

k_values = range(1,10)
plt.figure(figsize=(15, 5))

plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_nn, marker="o", label="Vanilla")
plt.plot(k_values, micro_f1_scores_nn_aug, marker="o", label="Data Augmentation")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(k_values, macro_f1_scores_nn, marker="o", label="Vanilla")
plt.plot(k_values, macro_f1_scores_nn_aug, marker="o", label="Data Augmentation")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()
plt.grid(True)

plt.grid(True)
plt.savefig('grafica2.png', bbox_inches='tight')
plt.show()

## Fine Tunning

In [ ]:
from keras_nlp.models import BertClassifier
import keras

BERT = BertClassifier.from_preset(
        "bert_base_en_uncased",
        num_classes=20,
        load_weights=True,
        preprocessor=keras_nlp.models.BertPreprocessor.from_preset(
            "bert_base_en_uncased",
            sequence_length=256
    )
    )
    
keras.Model.summary(BERT)

Y ahora la del modelo **distil_bert_base_en_uncased**

In [ ]:
from keras_nlp.models import DistilBertClassifier

DISILBERT = DistilBertClassifier.from_preset(
        "distil_bert_base_en_uncased",
        num_classes=20,
        preprocessor=keras_nlp.models.DistilBertPreprocessor.from_preset(
            "distil_bert_base_en_uncased",
            sequence_length=256
        )
    )
keras.Model.summary(DISILBERT)

In [ ]:
import keras_nlp
from keras_nlp.models import BertClassifier

def fine_tuning_BERT(X,y,X_val =np.array([]), y_val= np.array([]),train_backbone=True, learning_rate=1e-5, epochs =10, batch_size= 10, early_stopping=True,verbose=2):
    num_classes = np.unique(y).size
    model = BertClassifier.from_preset(
        "bert_base_en_uncased",
        num_classes=num_classes,
        load_weights=True,
        preprocessor=keras_nlp.models.BertPreprocessor.from_preset(
            "bert_base_en_uncased",
            sequence_length=256
    )
    )
    
    model.backbone.trainable = train_backbone

    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=["accuracy"]
    )
    if early_stopping:
        
        early_stopping = EarlyStopping(
            monitor='val_loss',       
            patience=3,              
           restore_best_weights=True 
        )

        
        history = model.fit(X, y,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stopping],    
            verbose=verbose)
            
    else:
            
        history = model.fit(X, y,
              epochs=epochs,
              batch_size=batch_size,
              verbose= verbose)
    return model

Misma estructura de entrenamiento para el modelo **disilbert_en_base_uncased**

In [ ]:
import keras_nlp
from keras_nlp.models import DistilBertClassifier

def fine_tuning_DISILBERT(X,y,X_val =np.array([]), y_val= np.array([]),train_backbone=True, learning_rate=1e-5, epochs =10, batch_size= 10, early_stopping=True,verbose=2):
 
    num_classes = np.unique(y).size
    model = DistilBertClassifier.from_preset(
        "distil_bert_base_en_uncased",
        num_classes=num_classes,
        preprocessor=keras_nlp.models.DistilBertPreprocessor.from_preset(
            "distil_bert_base_en_uncased",
            sequence_length=256
        )
    )
    model.backbone.trainable = train_backbone

    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=["accuracy"]
    )
    if early_stopping:
        
        early_stopping = EarlyStopping(
            monitor='val_loss',       
            patience=3,              
           restore_best_weights=True 
        )

        
        history = model.fit(X, y,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stopping],    
            verbose=verbose)
            
    else:
            
        history = model.fit(X, y,
              epochs=epochs,
              batch_size=batch_size,
              verbose= verbose)
    return model

### Evaluación del modelo

In [ ]:

#micro_f1_scores_BERT, macro_f1_scores_BERT = kscores(fine_tuning_BERT, range(1,10))


In [ ]:

#micro_f1_scores_DISILBERT, macro_f1_scores_DISILBERT = kscores(fine_tuning_DISILBERT, range(1,10))

In [ ]:
#Resultados de los últimos test
micro_f1_scores_BERT, macro_f1_scores_BERT = [0.07474774296335635, 0.08908656399362719, 0.1417950079660117, 0.19915029208709506, 0.17591609134360064, 0.27270313329792883, 0.35156664896441847, 0.3509028146574615, 0.33736059479553904] ,  [0.04027719849451687, 0.06102746506414829, 0.13008898039358996, 0.16468804778157525, 0.14832598026328186, 0.24451384638588397, 0.3306677370019278, 0.3042622734370731, 0.3077027501079125]
micro_f1_scores_DISILBERT , macro_f1_scores_DISILBERT = [0.12055231014338821, 0.15546999468932554, 0.27071163037705787, 0.32474774296335635, 0.34532660647902286, 0.45472650026553374, 0.42166755177907594, 0.4563197026022305, 0.4475570897503983] ,  [0.07974489395626465, 0.10849473517074207, 0.22181021633400735, 0.2817908085193318, 0.27470725890795517, 0.4089665105383931, 0.368159492236481, 0.40967819182998494, 0.4250614755783567]

In [ ]:
print(micro_f1_scores_BERT,", ",macro_f1_scores_BERT )
print(micro_f1_scores_DISILBERT,", ",macro_f1_scores_DISILBERT )

In [ ]:
import matplotlib.pyplot as plt
k_values = range(2,11)
plt.figure(figsize=(15, 5))

plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_nn,marker="o",label="Vanilla")
plt.plot(k_values, micro_f1_scores_nn_aug,marker="o", label="Data Augmentation")
plt.plot(k_values, micro_f1_scores_BERT, marker="o",label="BERT")
plt.plot(k_values, micro_f1_scores_DISILBERT,marker="o",  label="DISILBERT")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.grid(True)
plt.legend()

plt.subplot(1,2,2)
plt.plot(k_values, macro_f1_scores_nn, marker="o",label="Vanilla")
plt.plot(k_values, macro_f1_scores_nn_aug, marker="o",label="Data Augmentation")
plt.plot(k_values, macro_f1_scores_BERT, marker="o",label="BERT")
plt.plot(k_values, macro_f1_scores_DISILBERT,marker="o", label="DISILBERT")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.grid(True)
plt.legend()
plt.grid(True)
plt.savefig('grafica3.png', bbox_inches='tight')
plt.show()

## In context learning

In [ ]:
tags_banking = [
  { "id": 0, "intent": "activate_my_card" },
  { "id": 1, "intent": "age_limit" },
  { "id": 2, "intent": "apple_pay_or_google_pay" },
  { "id": 3, "intent": "atm_support" },
  { "id": 4, "intent": "automatic_top_up" },
  { "id": 5, "intent": "balance_not_updated_after_bank_transfer" },
  { "id": 6, "intent": "balance_not_updated_after_cheque_or_cash_deposit" },
  { "id": 7, "intent": "beneficiary_not_allowed" },
  { "id": 8, "intent": "cancel_transfer" },
  { "id": 9, "intent": "card_about_to_expire" },
  { "id": 10, "intent": "card_acceptance" },
  { "id": 11, "intent": "card_arrival" },
  { "id": 12, "intent": "card_delivery_estimate" },
  { "id": 13, "intent": "card_linking" },
  { "id": 14, "intent": "card_not_working" },
  { "id": 15, "intent": "card_payment_fee_charged" },
  { "id": 16, "intent": "card_payment_not_recognised" },
  { "id": 17, "intent": "card_payment_wrong_exchange_rate" },
  { "id": 18, "intent": "card_swallowed" },
  { "id": 19, "intent": "cash_withdrawal_charge" },
  { "id": 20, "intent": "cash_withdrawal_not_recognised" },
  { "id": 21, "intent": "change_pin" },
  { "id": 22, "intent": "compromised_card" },
  { "id": 23, "intent": "contactless_not_working" },
  { "id": 24, "intent": "country_support" },
  { "id": 25, "intent": "declined_card_payment" },
  { "id": 26, "intent": "declined_cash_withdrawal" },
  { "id": 27, "intent": "declined_transfer" },
  { "id": 28, "intent": "direct_debit_payment_not_recognised" },
  { "id": 29, "intent": "disposable_card_limits" },
  { "id": 30, "intent": "edit_personal_details" },
  { "id": 31, "intent": "exchange_charge" },
  { "id": 32, "intent": "exchange_rate" },
  { "id": 33, "intent": "exchange_via_app" },
  { "id": 34, "intent": "extra_charge_on_statement" },
  { "id": 35, "intent": "failed_transfer" },
  { "id": 36, "intent": "fiat_currency_support" },
  { "id": 37, "intent": "get_disposable_virtual_card" },
  { "id": 38, "intent": "get_physical_card" },
  { "id": 39, "intent": "getting_spare_card" },
  { "id": 40, "intent": "getting_virtual_card" },
  { "id": 41, "intent": "lost_or_stolen_card" },
  { "id": 42, "intent": "lost_or_stolen_phone" },
  { "id": 43, "intent": "order_physical_card" },
  { "id": 44, "intent": "passcode_forgotten" },
  { "id": 45, "intent": "pending_card_payment" },
  { "id": 46, "intent": "pending_cash_withdrawal" },
  { "id": 47, "intent": "pending_top_up" },
  { "id": 48, "intent": "pending_transfer" },
  { "id": 49, "intent": "pin_blocked" },
  { "id": 50, "intent": "receiving_money" },
  { "id": 51, "intent": "Refund_not_showing_up" },
  { "id": 52, "intent": "request_refund" },
  { "id": 53, "intent": "reverted_card_payment" },
  { "id": 54, "intent": "supported_cards_and_currencies" },
  { "id": 55, "intent": "terminate_account" },
  { "id": 56, "intent": "top_up_by_bank_transfer_charge" },
  { "id": 57, "intent": "top_up_by_card_charge" },
  { "id": 58, "intent": "top_up_by_cash_or_cheque" },
  { "id": 59, "intent": "top_up_failed" },
  { "id": 60, "intent": "top_up_limits" },
  { "id": 61, "intent": "top_up_reverted" },
  { "id": 62, "intent": "topping_up_by_card" },
  { "id": 63, "intent": "transaction_charged_twice" },
  { "id": 64, "intent": "transfer_fee_charged" },
  { "id": 65, "intent": "transfer_into_account" },
  { "id": 66, "intent": "transfer_not_received_by_recipient" },
  { "id": 67, "intent": "transfer_timing" },
  { "id": 68, "intent": "unable_to_verify_identity" },
  { "id": 69, "intent": "verify_my_identity" },
  { "id": 70, "intent": "verify_source_of_funds" },
  { "id": 71, "intent": "verify_top_up" },
  { "id": 72, "intent": "virtual_card_not_working" },
  { "id": 73, "intent": "visa_or_mastercard" },
  { "id": 74, "intent": "why_verify_identity" },
  { "id": 75, "intent": "wrong_amount_of_cash_received" },
  { "id": 76, "intent": "wrong_exchange_rate_for_cash_withdrawal" }
]
tag_names_banking = [i["intent"] for i in tags_banking]

In [ ]:
tags_20_newsgroup = [
    { "id": 0, "intent": "alt.atheism" },
    { "id": 1, "intent": "comp.graphics" },
    { "id": 2, "intent": "comp.os.ms-windows.misc" },
    { "id": 3, "intent": "comp.sys.ibm.pc.hardware" },
    { "id": 4, "intent": "comp.sys.mac.hardware" },
    { "id": 5, "intent": "comp.windows.x" },
    { "id": 6, "intent": "misc.forsale" },
    { "id": 7, "intent": "rec.autos" },
    { "id": 8, "intent": "rec.motorcycles" },
    { "id": 9, "intent": "rec.sport.baseball" },
    { "id": 10, "intent": "rec.sport.hockey" },
    { "id": 11, "intent": "sci.crypt" },
    { "id": 12, "intent": "sci.electronics" },
    { "id": 13, "intent": "sci.med" },
    { "id": 14, "intent": "sci.space" },
    { "id": 15, "intent": "soc.religion.christian" },
    { "id": 16, "intent": "talk.politics.guns" },
    { "id": 17, "intent": "talk.politics.mideast" },
    { "id": 18, "intent": "talk.politics.misc" },
    { "id": 19, "intent": "talk.religion.misc" }
]

tag_names_20_newsgroup = [i["intent"] for i in tags_20_newsgroup]

In [ ]:
system_message = """
Task: You are {role_description}.
Your sole responsibility is to classify the intent of an incoming text into one of the following categories. 

Categories:
{categories}

Constraints:
- Your only task is to reveal the intent of the text, no context or explanation required.
- You must ONLY return the name of the intent of the text choosing from the list above.
- You must NOT provide introductions, explanations, analysis, greetings, punctuation or any other text in your respone.
- If the intent is ambiguous, choose the most statistically likely category based on the examples but only return ONE category.
- If your response violates any of this constraints, the clasification will be consdered wrong, no matter if you guessed the class correctly.

Examples:
{examples}
"""

In [ ]:
"""
##### 1. Instalación de librerías necesarias

!apt-get install -y zstd
!pip install langchain-ollama langchain-core

#### 2. Instalación de Ollama (Script oficial)

#!curl -fsSL https://ollama.com/install.sh | sh

#### 3. Lanzar servidor en segundo plano

#import subprocess
#import time
#import os

ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

"""

In [ ]:
#### Esperar a que el servidor arranque y descargar Llama 3.1
"""
!ollama pull llama3.1:8b
"""

In [ ]:
#Para desinstalar el servidor 
#ollama_process.terminate()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
BASE_URL = os.getenv("SERVER_BASE_URL")
API_KEY = os.getenv("OLLAMA_API_KEY")

In [ ]:
from langchain_openai import ChatOpenAI

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate)


def create_system_promt(role_description, tag_names, system_promt, X=np.array([]), y=np.array([])):
    num_examples = X.size
    if num_examples > 0:
        example_list = [f"Text \n: {X[i]},\n Expected output \n: {tag_names[y[i]]}" for i in range(num_examples)]
        examples = "\n".join(example_list)
    else:
        examples = "No examples provided"
        
    categories_str = ", ".join(tag_names)
    base_prompt_template = SystemMessagePromptTemplate.from_template(system_promt)
    system_message_prompt = base_prompt_template.format(
        role_description=role_description, 
        categories=categories_str, 
        examples=examples
    )
    return system_message_prompt

def run_intent_classification(X_test, system_message_prompt, tag_names,model="llama3.1:8b", temperature=0,base_url=None, api_key=None):
    human_template = "Classify the intent of the following text: \n {text}"
    human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message_prompt,
        human_message_prompt
    ])

    # El servidor utiliza webui, que se comunica de la misma forma que la API de OpenAI
    chat_model = ChatOpenAI(
        model=model, 
        temperature=temperature,
        openai_api_base=base_url, 
        api_key=api_key
    )
    # Si se quiere usar Ollama en local: 
    # chat_model = ChatOllama(model=model, temperature=temperature)

    responses = []
    for i,X_test_item in enumerate(X_test):
        formatted_prompt = chat_prompt.format_prompt(text=X_test_item).to_messages()
        response = chat_model.invoke(formatted_prompt)
        response_text = response.content.strip()
        if response_text in tag_names:
            response_index = tag_names.index(response_text)
        else:
            response_index = len(tag_names) +1
        if i<5:
            print(f"Text:  {X_test_item}, response:  {response.content} ,class:  {response.content.strip()}")

        responses.append(response_index)

    return responses

### Evaluación del modelo

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

def promt_scores(k_values, tag_names, system_promt, role,base_url=None, api_key=None):
    micro_f1_scores = []
    macro_f1_scores = []

    for k in k_values:
        train = train_fs_list[k]
        X = train['text'].values
        y = train['label'].values


        X_test = test['text'].values
        y_test = test['label'].values

        system_message_prompt = create_system_promt(role,tag_names, system_promt,X, y)
        y_pred = run_intent_classification(X_test, system_message_prompt, tag_names, base_url=base_url, api_key=api_key)

        micro_f1 = f1_score(y_test, y_pred, average='micro')
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        micro_f1_scores.append(micro_f1)
        macro_f1_scores.append(macro_f1)

    return micro_f1_scores, macro_f1_scores

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

def promt_scores_zero(tag_names, system_promt, role,base_url=None, api_key=None):
    micro_f1_scores = []
    macro_f1_scores = []
    
    X_test = test['text'].values
    y_test = test['label'].values


    system_message_prompt = create_system_promt(role,tag_names, system_promt)
    y_pred = run_intent_classification(X_test, system_message_prompt, tag_names, base_url=base_url, api_key=api_key)

    micro_f1 = f1_score(y_test, y_pred, average='micro')
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    micro_f1_scores.append(micro_f1)
    macro_f1_scores.append(macro_f1)

    return micro_f1_scores, macro_f1_scores

In [ ]:
role_newsgroup = "a sociologist studying the messages shared on the Internet"
role_banking = "an asistant for a banking customer service hot line"
#micro_f1_scores_promt_zero, macro_f1_scores_promt_zero = promt_scores_zero(tag_names_20_newsgroup, system_message, role_newsgroup,base_url=BASE_URL, api_key=API_KEY)

In [ ]:
#micro_f1_scores_promt_bad, macro_f1_scores_promt_bad = promt_scores(range(1,4),tag_names_20_newsgroup, system_message, role_newsgroup,base_url=BASE_URL, api_key=API_KEY)

In [ ]:
micro_f1_scores_promt_bad, macro_f1_scores_promt_bad = ([0.3433351035581519, 0.4195432819968136 ,0.369888475836431259]+[0.0]*6) , ([0.3267146958921656, 0.4173130346142294, 0.36563951034330316] + [0.0]*6)


In [ ]:
print(micro_f1_scores_promt_bad,",", macro_f1_scores_promt_bad)

In [ ]:
#Resultados del anterior test
micro_f1_scores_promt_zero, macro_f1_scores_promt_zero = [0.5386351566648965]*9 , [0.49962380323731703]*9


In [ ]:
#Mostramos resultados
print(micro_f1_scores_promt_zero,",", macro_f1_scores_promt_zero)

### Verbosidad del modelo: RAG

In [ ]:
#Si tiene ollama instalado en local puede hacer uso del siguiente embedding
#!ollama pull nomic-embed-text

In [ ]:
import numpy as np
from langchain_openai import ChatOpenAI
from langchain_openai import  OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

def create_system_promt_RAG(role_description, tag_names, system_promt, X=np.array([]), y=np.array([])):
 
    categories_str = ", ".join(tag_names)
    base_prompt_template = SystemMessagePromptTemplate.from_template(system_promt)
    system_message_prompt = base_prompt_template.format(
        role_description=role_description, 
        categories=categories_str, 
        examples="{examples}"
    )
    return system_message_prompt

def run_intent_classification_RAG(X_test,X_train,y_train, system_message_prompt,embedding_model, tag_names,model="llama3.1:8b", temperature=0,k=5,base_url=None,api_key=None):
    human_template = "Classify the intent of the following text: \n {text}"
    #Si se cuenta con embeddings en el servidor
    embeddings = OpenAIEmbeddings(
        model=embedding_model,
        openai_api_key=api_key,
        openai_api_base=base_url,
        tiktoken_enabled=False,
        check_embedding_ctx_length=False,
    )
    #Si se ejecuta el embedding en local
    #embeddings = OllamaEmbeddings(model=embedding_model) 

    docs = []
    for i in range(X_train.size):
        example_text = f"text: {X_train[i]}, category: {tag_names[y_train[i]]}"
        docs.append(Document(page_content=example_text))

    print(f"Indexing {len(docs)} training examples into FAISS...")

    vector_store = FAISS.from_documents(docs, embeddings)

    retriever = vector_store.as_retriever(search_kwargs={"k": k})
    human_template = "Classify the intent of the following text: {text}"
    human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message_prompt,
        human_message_prompt
    ])
    # El servidor utiliza webui, que se comunica de la misma forma que la API de OpenAI
    chat_model = ChatOpenAI(model=model, temperature=temperature,openai_api_base=base_url,  api_key=api_key)
    #Si se opta por un modelo instalado en local
    #chat_model = ChatOllama(model=model, temperature=temperature)

    responses = []
    print("Starting classification...")
    for i, X_test_item in enumerate(X_test):
        relevant_docs = retriever.invoke(X_test_item)

        if relevant_docs:
            examples_str = "\n".join([doc.page_content for doc in relevant_docs])
        else:
            examples_str = "No examples provided"

        formatted_prompt = chat_prompt.format_prompt(
            examples=examples_str,
            text=X_test_item
        ).to_messages()

        response = chat_model.invoke(formatted_prompt)
        response_text = response.content.strip()

        if response_text in tag_names:
            response_index = tag_names.index(response_text)
        else:
            response_index = np.unique(y).size +1

        if i < 5:
            print(f"\n--- Example {i+1} ---")
            print(f"Test Text: {X_test_item}")
            print(f"Retrieved via FAISS:\n{examples_str}")
            print(f"Prediction: {response_text}")

        responses.append(response_index)

    return responses

Para evaluar el modelo de RAG se ha creado la siguiente función:

In [ ]:
def promt_scores_RAG(k_values, tag_names, system_promt, role, embedding_model, base_url=None, api_key=None):
    micro_f1_scores = []
    macro_f1_scores = []

    for k in k_values:
        train = train_fs_list[k]
        X = train['text'].values
        y = train['label'].values

        X_test = test['text'].values
        y_test = test['label'].values

        system_message_prompt = create_system_promt_RAG(role, tag_names, system_promt)
        
        y_pred = run_intent_classification_RAG(
            X_test, X, y, system_message_prompt, 
            tag_names=tag_names, 
            embedding_model=embedding_model, 
            base_url=base_url, 
            api_key=api_key
        )

        micro_f1 = f1_score(y_test, y_pred, average='micro')
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        micro_f1_scores.append(micro_f1)
        macro_f1_scores.append(macro_f1)
        
    return micro_f1_scores, macro_f1_scores

In [ ]:
"""
micro_f1_scores_promt_RAG, macro_f1_scores_promt_RAG = promt_scores_RAG(
    k_values=range(1, 10),
    tag_names=tag_names_20_newsgroup, 
    system_promt=system_message, 
    role=role_newsgroup,
    embedding_model="mxbai-embed-large:335m",
    base_url=BASE_URL, 
    api_key=API_KEY
)
"""

In [ ]:
micro_f1_scores_promt_RAG, macro_f1_scores_promt_RAG = [0.5302708443972385, 0.5335900159320234, 0.5339883165161976, 0.526420605416888, 0.5335900159320234, 0.5318640467339352, 0.531465746149761, 0.5318640467339352, 0.5335900159320234] , [0.49111627883838743, 0.4949399638090258, 0.4943154634542336, 0.48785160944135386, 0.4935315790117089, 0.49289589628109387, 0.4917506884705354, 0.49165537468812315, 0.4952203536942378]
print(micro_f1_scores_promt_RAG, ",", macro_f1_scores_promt_RAG )


In [ ]:
plt.figure(figsize=(15, 5))
k_values= range(2,11)
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_promt_zero, marker="o", label="Zero-shot LLM")
plt.plot(k_values, micro_f1_scores_promt_bad, marker="o", label="LLM Sin RAG")
plt.plot(k_values, micro_f1_scores_promt_RAG, marker="o", label="LLM RAG")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(k_values, macro_f1_scores_promt_zero, marker="o", label="Zero-shot LLM")
plt.plot(k_values, macro_f1_scores_promt_bad, marker="o", label="LLM Sin RAG")
plt.plot(k_values, micro_f1_scores_promt_RAG, marker="o", label="LLM RAG")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()
plt.grid(True)
plt.savefig('grafica4.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
k_values= range(2,11)
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_nn,marker="o",label="Vanilla")
plt.plot(k_values, micro_f1_scores_nn_aug,marker="o", label="Data Augmentation")
plt.plot(k_values, micro_f1_scores_BERT, marker="o",label="BERT")
plt.plot(k_values, micro_f1_scores_DISILBERT,marker="o",  label="DISILBERT")
plt.plot(k_values, micro_f1_scores_promt_zero, marker="o", label="Zero-shot LLM")
plt.plot(k_values, micro_f1_scores_promt_bad, marker="o", label="LLM Sin RAG")
plt.plot(k_values, micro_f1_scores_promt_RAG, marker="o", label="LLM RAG")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(k_values, macro_f1_scores_nn,marker="o",label="Vanilla")
plt.plot(k_values, macro_f1_scores_nn_aug,marker="o", label="Data Augmentation")
plt.plot(k_values, macro_f1_scores_BERT, marker="o",label="BERT")
plt.plot(k_values, macro_f1_scores_DISILBERT,marker="o",  label="DISILBERT")
plt.plot(k_values, macro_f1_scores_promt_zero, marker="o", label="Zero-shot LLM")
plt.plot(k_values, macro_f1_scores_promt_bad, marker="o", label="LLM Sin RAG")
plt.plot(k_values, micro_f1_scores_promt_RAG, marker="o", label="LLM RAG")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")

plt.grid(True)
plt.savefig('grafica5.png', bbox_inches='tight')
plt.show()

## Metric Learning

### Redes Siamesas (Siamese Neural Networks)

In [ ]:
class SiameseNetwork(tf.keras.Model):
    def __init__(self, encoder, **kwargs):
        super(SiameseNetwork, self).__init__(**kwargs)
        self.encoder = encoder

    def call(self, inputs):
        input_a, input_b = inputs

        embedding_a = self.encoder(input_a)
        embedding_b = self.encoder(input_b)

        a_norm = tf.nn.l2_normalize(embedding_a, axis=1)
        b_norm = tf.nn.l2_normalize(embedding_b, axis=1)

        distance = tf.sqrt(tf.reduce_sum(tf.square(a_norm - b_norm), axis=1) + 1e-7)
        
        return distance

In [ ]:
def create_SNN(num_layers=2, layers_size=512, activation='relu', embedding=True):
    
    encoder = models.Sequential()
    if embedding:
        encoder.add(
            hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=False,
            input_shape=[],
            dtype=tf.string
        )
    )

    for _ in range(num_layers):
        encoder.add(layers.Dense(layers_size, activation=activation))

    model = SiameseNetwork(encoder=encoder)

    return model

.

In [ ]:
def predict_one_shot_snn(model, X_q, X_sup, y_sup):
    predictions = []
    X_sup_tensor = tf.constant(X_sup, dtype=tf.string)
    for q in X_q:
        q_rep = tf.constant([q] * len(X_sup), dtype=tf.string)

        logits = model.predict([q_rep, X_sup_tensor], verbose=0)
        predictions.append(y_sup[np.argmin(logits)])
    return np.array(predictions)

In [ ]:
def create_pairs(X, y):
    x_a, x_b, y_p = [], [], []
    labels = np.unique(y)
    label_groups = [np.where(y == i)[0] for i in labels]
    for i in range(len(label_groups)):
        indices = label_groups[i]
        for j in range(len(indices) - 1):
            #Creamos un par de elementos de una misma clase
            x_a.append(X[indices[j]])
            x_b.append(X[indices[j+1]])
            y_p.append(1.0)
            #Creamos un par de elementos de clases distintas
            distinta =  random.choice(labels[labels != labels[i]])
            neg_candidates = np.where(y == distinta)[0]
            neg_idx = np.random.choice(neg_candidates)
            
            x_a.append(X[indices[j]])
            x_b.append(X[neg_idx])
            y_p.append(0.0)
    return np.array(x_a), np.array(x_b), np.array(y_p)


def contrastive_loss(y_true, y_pred, margin=1.0):
    y_true = tf.cast(y_true, tf.float32)
    square_pred = tf.square(y_pred)
    margin_square = tf.square(tf.maximum(margin - y_pred, 0))
    return tf.reduce_mean(y_true * square_pred + (1 - y_true) * margin_square)



In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

def siamese_scores(k_values, net_params=dict(),epochs=5,batch_size=20,learning_rate=1e-3,embedding=True):
    micro_f1_scores_untrained = []
    macro_f1_scores_untrained = []
    micro_f1_scores_trained = []
    macro_f1_scores_trained = []
    
    X_support = train_fs_list[0]['text'].values
    y_support = train_fs_list[0]['label'].values
    X_test = test['text'].values
    y_test = test['label'].values
    
    for k in k_values:
        train = train_fs_list[k]

        if embedding:
            
            X = train['text'].values
            X_test = test['text'].values
            
        else:
            X = TOKENIZER( train['text'].values)
            X_test = TOKENIZER(test['text'].values)


        y = train['label'].values
        y_test = test['label'].values
        
        model = create_SNN(**net_params)


        y_pred_1 = predict_one_shot_snn(model, X_test, X_support, y_support)

        xa, xb, yp = create_pairs(X, y)
        model.compile(loss=contrastive_loss, optimizer=optimizers.Adam(learning_rate=learning_rate),run_eagerly=True)
        
        

        model.fit(
            x=[tf.constant(xa), tf.constant(xb)],
            y=tf.constant(yp),
            epochs=epochs,
            batch_size=batch_size
        )
        y_pred_2 = predict_one_shot_snn(model, X_test, X_support, y_support)
                 

        micro_f1_1 = f1_score(y_test, y_pred_1, average='micro')
        macro_f1_1 = f1_score(y_test, y_pred_1, average='macro')
        micro_f1_scores_untrained.append(micro_f1_1)
        macro_f1_scores_untrained.append(macro_f1_1)
        
        micro_f1_2 = f1_score(y_test, y_pred_2, average='micro')
        macro_f1_2 = f1_score(y_test, y_pred_2, average='macro')
        micro_f1_scores_trained.append(micro_f1_2)
        macro_f1_scores_trained.append(macro_f1_2)
        
        

    return micro_f1_scores_untrained, macro_f1_scores_untrained, micro_f1_scores_trained, macro_f1_scores_trained


In [ ]:
#micro_f1_scores_snn_untrained, macro_f1_scores_snn_untrained, micro_f1_scores_snn_trained, macro_f1_scores_snn_trained = siamese_scores(range(1,10))

In [ ]:
micro_f1_scores_snn_untrained, macro_f1_scores_snn_untrained= [0.18680297397769516, 0.18839617631439193, 0.17578332448220924, 0.18281996813595328, 0.18348380244291024, 0.1854753053637812, 0.16768454593733403, 0.18813064259160914, 0.19038767923526287] , [0.17609571023278192, 0.1778731110468967, 0.15402863220792593, 0.1669930832039178, 0.17671038633077982, 0.1495642628719723, 0.1548863366385858, 0.17140942748676757, 0.17012688323750327]
micro_f1_scores_snn_trained, macro_f1_scores_snn_trained = [0.22703133297928837, 0.2781465746149761, 0.2794742432288901, 0.2622145512480085, 0.25517790759426445, 0.29221986192246413, 0.29328199681359535, 0.3032395114179501, 0.29660116834838024] , [0.17848338165671523, 0.28167658605675894, 0.2571155226459747, 0.23106123950101276, 0.24795405051779387, 0.27083300134224625, 0.28424105081047346, 0.2932746370314273, 0.2979765968054203]

In [ ]:
print(micro_f1_scores_snn_untrained,",", macro_f1_scores_snn_untrained)
print(micro_f1_scores_snn_trained,"," ,macro_f1_scores_snn_trained)

In [ ]:
plt.figure(figsize=(15, 5))
k_values= range(2,11)
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_snn_untrained,marker="o",label="Red Siamesa sin entrenamiento")
plt.plot(k_values, micro_f1_scores_snn_trained, marker="o", label="Red Siamesa con entrenamiento")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(k_values, macro_f1_scores_snn_untrained,marker="o",label="Red Siamesa sin entrenamiento")
plt.plot(k_values, micro_f1_scores_snn_trained, marker="o", label="Red Siamesa con entrenamiento")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()

plt.grid(True)
plt.savefig('grafica6.png', bbox_inches='tight')
plt.show()

### Redes Relacionales (Matching Neural Networks)

In [ ]:
import tensorflow as tf

class MatchingNetwork(tf.keras.Model):
    def __init__(self, encoder, **kwargs):
        super(MatchingNetwork, self).__init__(**kwargs)
        self.encoder = encoder

    def compute_similarities(self, query_embeddings, support_embeddings):
        """
        Calcula la similitud de coseno entre queries y cada ejemplo del support set.
        Actúa como el equivalente a 'compute_distances' en Prototypical.
        """
        query_norm = tf.nn.l2_normalize(query_embeddings, axis=-1)
        support_norm = tf.nn.l2_normalize(support_embeddings, axis=-1)
        return tf.matmul(query_norm, support_norm, transpose_b=True)

    def call(self, support_set, support_labels, query_set):
        """
        Retorna los logits basados en la suma de similitudes por clase.
        """
        num_classes = tf.size(tf.unique(support_labels)[0])
        support_embeddings = self.encoder(support_set)
        query_embeddings = self.encoder(query_set)
        support_embeddings = tf.cast(support_embeddings, tf.float32)
        query_embeddings = tf.cast(query_embeddings, tf.float32)

        similarities = self.compute_similarities(query_embeddings, support_embeddings)

        support_labels_one_hot = tf.one_hot(support_labels, depth=num_classes)

        logits = tf.matmul(similarities, support_labels_one_hot)

        return logits

    def predict(self, support_set, support_labels, query_set):
        logits = self.call(support_set, support_labels, query_set)
        return tf.nn.softmax(tf.cast(logits, tf.float32))

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization

def create_MNN(num_layers=2, layers_size=512, activation='relu', embedding=True):
    
    encoder = models.Sequential()

    if embedding:
        encoder.add(
            hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=False,
            input_shape=[],
            dtype=tf.string
        )
    )

    for _ in range(num_layers):
        encoder.add(layers.Dense(layers_size, activation=activation))

    model = MatchingNetwork(encoder=encoder)

    return model

**Primer Algoritmo: entrenamiento episódico completo**:



In [ ]:
def train_step(model, X_sup, y_sup, X_query, y_query, optimizer, loss_fn):

    with tf.GradientTape() as tape:
        distances = model(X_sup, y_sup, X_query)
        loss = loss_fn(y_query, distances)

    variables = model.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return loss, distances



def train_net(model, X, y, epochs=10, learning_rate=1e-4 , support_size=0.5):
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    for epoch in range(epochs):
        X_sup, X_query, y_sup, y_query = train_test_split(X, y, test_size=support_size, stratify=y, random_state=42)
        X_sup, X_query = tf.constant(X_sup), tf.constant(X_query)
        y_sup, y_query = tf.constant(y_sup, dtype=tf.int32), tf.constant(y_query, dtype=tf.int32)
        loss, logits = train_step(model, X_sup, y_sup, X_query, y_query,optimizer, loss_fn)

        preds = tf.argmax(logits, axis=1, output_type=tf.int32)
        acc = tf.reduce_mean(tf.cast(tf.equal(preds, y_query), tf.float32))

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"Epoca {epoch:02d} | Loss: {loss:.4f} | Accuracy: {acc:.2%}")

    return model

**Segundo Algortimo:  entrenamiento episódico por clases**:


In [ ]:
def train_step2(model, X_sup, y_sup, X_query, y_query, num_classes, optimizer, loss_fn):
    with tf.GradientTape() as tape:
        distances = model(X_sup, y_sup, X_query)
        loss = loss_fn(y_query, distances)

    vars = model.trainable_variables
    gradients = tape.gradient(loss, vars)
    optimizer.apply_gradients(zip(gradients, vars))
    return loss, distances

def get_episode(X, y, batch_classes, support_size):
    all_classes = np.unique(y)
    if all_classes.size > batch_classes:
        sampled_classes = np.random.choice(all_classes, size=batch_classes, replace=False)
    else:
        sampled_classes = all_classes

    X_sup, y_sup, X_q, y_q = [], [], [], []

    for i, cls in enumerate(sampled_classes):
        indices = np.where(y == cls)[0]
        n_total = indices.size
        n_support = max(1, int(n_total * support_size))
        if n_support == n_total:
            n_support = n_total - 1
        n_query = n_total - n_support

        np.random.shuffle(indices)

        sup_idx = indices[:n_support]
        q_idx = indices[n_support:]

        X_sup.extend(X[sup_idx])
        X_q.extend(X[q_idx])
        y_sup.extend([i] * n_support)
        y_q.extend([i] * n_query)

    mask_remaining = ~np.isin(y, sampled_classes)
    X_remaining = X[mask_remaining]
    y_remaining = y[mask_remaining]

    return (X_remaining, y_remaining, tf.constant(X_sup), tf.constant(y_sup, dtype=tf.int32),
            tf.constant(X_q), tf.constant(y_q, dtype=tf.int32), batch_classes)

def train_net2(model, X, y, epochs=10, learning_rate=1e-4, batch_classes=5, support_size=0.5):
    num_classes = np.unique(y).size
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    for epoch in range(epochs):
        epoch_loss = 0
        epoch_acc = 0
        steps_per_epoch = 0
        X_iter = X.copy()
        y_iter = y.copy()
        while (X_iter.size > 0):
            X_iter, y_iter, X_s, y_s, X_q, y_q, n_c = get_episode(X_iter, y_iter, batch_classes, support_size)

            loss, logits = train_step2(model, X_s, y_s, X_q, y_q, n_c, optimizer, loss_fn)

            preds = tf.argmax(logits, axis=1, output_type=tf.int32)
            acc = tf.reduce_mean(tf.cast(tf.equal(preds, y_q), tf.float32))

            steps_per_epoch += 1
            epoch_loss += loss
            epoch_acc += acc

        if epoch % 5 == 0 or epoch == epochs -1:
            print(f"Epoca {epoch+1:02d} | Loss: {epoch_loss/steps_per_epoch:.4f} | "
                  f"Avg Acc: {epoch_acc/steps_per_epoch:.2%}")

    return model

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report


def matching_scores(k_values,train_algorithm, net_params=dict(),train_params=dict(),embedding=True):
    micro_f1_scores_untrained = []
    macro_f1_scores_untrained = []
    micro_f1_scores_trained = []
    macro_f1_scores_trained = []
    for k in k_values:
        train = train_fs_list[k]

        if embedding:
            
            X = train['text'].values
            X_test = test['text'].values
            
        else:
            X = TOKENIZER( train['text'].values)
            X_test = TOKENIZER(test['text'].values)


        y = train['label'].values
        y_test = test['label'].values
        y_test = tf.constant(y_test, dtype=tf.int32)
        
        model = create_MNN(**net_params)

        y_pred_1_proba = model.predict(query_set=X_test, support_set=X, support_labels=y)
        y_pred_1 = np.argmax(y_pred_1_proba, axis=1)

        
        train_algorithm(model=model,X=X,y=y, **train_params)
        y_pred_2_proba = model.predict(query_set=X_test, support_set=X, support_labels=y)
        y_pred_2 = np.argmax(y_pred_2_proba, axis=1)
                   
        micro_f1_1 = f1_score(y_test, y_pred_1, average='micro')
        macro_f1_1 = f1_score(y_test, y_pred_1, average='macro')
        print(micro_f1_1)
        print(macro_f1_1)
        micro_f1_scores_untrained.append(micro_f1_1)
        macro_f1_scores_untrained.append(macro_f1_1)
        micro_f1_2 = f1_score(y_test, y_pred_2, average='micro')
        macro_f1_2 = f1_score(y_test, y_pred_2, average='macro')
        print(micro_f1_2)
        print(macro_f1_2)
        micro_f1_scores_trained.append(micro_f1_2)
        macro_f1_scores_trained.append(macro_f1_2)
        
        

    return micro_f1_scores_untrained, macro_f1_scores_untrained, micro_f1_scores_trained, macro_f1_scores_trained


In [ ]:
#micro_f1_scores_mnn_untrained1, macro_f1_scores_mnn_untrained1, micro_f1_scores_mnn_trained1, macro_f1_scores_mnn_trained1 = matching_scores(range(1,10),train_net)

In [ ]:
micro_f1_scores_mnn_untrained1, macro_f1_scores_mnn_untrained1 =[0.2454859267126925, 0.23937865108868828, 0.29235262878385554, 0.27296866702071165, 0.2825278810408922, 0.30443441317047265, 0.34014869888475835, 0.3106744556558683, 0.35236325013276687] , [0.22890143377453556, 0.21786020940199508, 0.2635995712963445, 0.2587802348764035, 0.24884222904961067, 0.28186034678342414, 0.32575244125773, 0.2781675127666905, 0.31926192143332777]
micro_f1_scores_mnn_trained1, macro_f1_scores_mnn_trained1 = [0.264737121614445, 0.2575677110993096, 0.3181093998937865, 0.29726500265533723, 0.3252788104089219, 0.338688263409453, 0.3603292618162507, 0.3503717472118959, 0.37745618693574085] , [0.24949222044549887, 0.24720842625706502, 0.2983605543806517, 0.28688706565733657, 0.2999325949549917, 0.3189691323071179, 0.3439673658007337, 0.3181930155012904, 0.35040091639478715]

In [ ]:

print(micro_f1_scores_mnn_untrained1,",", macro_f1_scores_mnn_untrained1) 
print(micro_f1_scores_mnn_trained1,",", macro_f1_scores_mnn_trained1)

In [ ]:
#micro_f1_scores_mnn_untrained2, macro_f1_scores_mnn_untrained2, micro_f1_scores_mnn_trained2, macro_f1_scores_mnn_trained2 = matching_scores(range(1,10),train_net2)

In [ ]:
micro_f1_scores_mnn_untrained2, macro_f1_scores_mnn_untrained2 = [0.2454859267126925, 0.23937865108868828, 0.29235262878385554, 0.27296866702071165, 0.2825278810408922, 0.30443441317047265, 0.34014869888475835, 0.3106744556558683, 0.35236325013276687] , [0.22890143377453556, 0.21786020940199508, 0.2635995712963445, 0.2587802348764035, 0.24884222904961067, 0.28186034678342414, 0.32575244125773, 0.2781675127666905, 0.31926192143332777]
micro_f1_scores_mnn_trained2, macro_f1_scores_mnn_trained2 = [0.2736325013276686, 0.27482740308019116, 0.3278013807753585, 0.3074880509824748, 0.3441317047265003, 0.35993096123207646, 0.36205523101433884, 0.3587360594795539, 0.38090812533191715] , [0.26007482346342287, 0.2659538955332706, 0.3105780509769728, 0.29561651521988774, 0.32508702628306535, 0.3438557750878969, 0.3401952172093473, 0.32444139761919355, 0.35789388079387546]

In [ ]:

print(micro_f1_scores_mnn_untrained2,",", macro_f1_scores_mnn_untrained2) 
print(micro_f1_scores_mnn_trained2,",", macro_f1_scores_mnn_trained2)

In [ ]:
plt.figure(figsize=(15, 5))
k_values= range(2,11)
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_mnn_untrained1,marker="o",label="Red Matching sin entrenamiento")
plt.plot(k_values, micro_f1_scores_mnn_trained1, marker="o", label="Red Matching con entrenamiento episódico")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(k_values, macro_f1_scores_mnn_untrained1,marker="o",label="Red Matching sin entrenamiento")
plt.plot(k_values, macro_f1_scores_mnn_trained1, marker="o", label="Red Matching con entrenamiento episódico")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()

plt.grid(True)
plt.savefig('grafica7.png', bbox_inches='tight')
plt.show()

In [ ]:



plt.figure(figsize=(15, 5))
k_values= range(2,11)
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_mnn_untrained2,marker="o",label="Red Matching sin entrenamiento")
plt.plot(k_values, micro_f1_scores_mnn_trained1, marker="o", label="Red Matching con entrenamiento")
plt.plot(k_values, micro_f1_scores_mnn_trained2, marker="o", label="Red Matching con entrenamiento por clases")

plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)

plt.plot(k_values, macro_f1_scores_mnn_untrained2,marker="o",label="Red Matching sin entrenamiento")
plt.plot(k_values, macro_f1_scores_mnn_trained1, marker="o", label="Red Matching con entrenamiento")
plt.plot(k_values, macro_f1_scores_mnn_trained2, marker="o", label="Red Matching con entrenamiento por clases")

plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()

plt.grid(True)
plt.savefig('grafica8.png', bbox_inches='tight')
plt.show()

### Redes Prototípicas

In [ ]:
class PrototypicalNetwork(tf.keras.Model):
    def __init__(self, encoder, **kwargs):
        super(PrototypicalNetwork, self).__init__(**kwargs)
        self.encoder = encoder
        self.prototypes = None

    def compute_prototypes(self, support_embeddings, support_labels, num_classes):
  
        prototypes = []

        for c in range(num_classes):
            class_mask = tf.equal(support_labels, c)
            class_embeddings = tf.boolean_mask(support_embeddings, class_mask)
            prototype = tf.reduce_mean(class_embeddings, axis=0)
            prototypes.append(prototype)
        return tf.stack(prototypes)

    def compute_distances(self, query_embeddings, prototypes):
        
        q_expanded = tf.expand_dims(query_embeddings, axis=1)
        p_expanded = tf.expand_dims(prototypes, axis=0)

        distances = tf.reduce_sum(tf.square(q_expanded - p_expanded), axis=2)
    
        return -distances

    def call(self, support_set, support_labels, query_set):

        
        num_classes = tf.size(tf.unique(support_labels)[0])
        support_embeddings = self.encoder(support_set)
        support_embeddings = tf.nn.l2_normalize(support_embeddings, axis=-1)
        query_embeddings = self.encoder(query_set)
        prototypes = self.compute_prototypes(support_embeddings, support_labels, num_classes)
        distances = self.compute_distances(query_embeddings, prototypes)

        return tf.cast(distances, tf.float32)

    def predict(self, support_set, support_labels, query_set):
        logits = self.call(support_set, support_labels, query_set)
        return tf.nn.softmax(tf.cast(logits, tf.float32))

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization

def create_PNN(num_layers=2, layers_size=512, activation='relu', embedding=True):
    
    encoder = models.Sequential()

    if embedding:
        encoder.add(
            hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=False,
            input_shape=[],
            dtype=tf.string
        )
    )

    for _ in range(num_layers):
        encoder.add(layers.Dense(layers_size, activation=activation))

    model = PrototypicalNetwork(encoder=encoder)

    return model

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

def prototypical_scores(k_values,train_algorithm, net_params=dict(),train_params=dict(),embedding=True):
    micro_f1_scores_untrained = []
    macro_f1_scores_untrained = []
    micro_f1_scores_trained = []
    macro_f1_scores_trained = []

    for k in k_values:
        train = train_fs_list[k]

        if embedding:
            
            X = train['text'].values
            X_test = test['text'].values
            
        else:
            X = TOKENIZER( train['text'].values)
            X_test = TOKENIZER(test['text'].values)


        y = train['label'].values
        y_val = val['label'].values
        y_test = test['label'].values
        y_test = tf.constant(y_test, dtype=tf.int32)
        
        model = create_PNN(**net_params)

        y_pred_1_proba = model.predict(query_set=X_test, support_set=X, support_labels=y)
        y_pred_1 = np.argmax(y_pred_1_proba, axis=1)

        
        train_algorithm(model=model,X=X,y=y, **train_params)
        y_pred_2_proba = model.predict(query_set=X_test, support_set=X, support_labels=y)
        y_pred_2 = np.argmax(y_pred_2_proba, axis=1)
                   
        micro_f1_1 = f1_score(y_test, y_pred_1, average='micro')
        macro_f1_1 = f1_score(y_test, y_pred_1, average='macro')

        micro_f1_scores_untrained.append(micro_f1_1)
        macro_f1_scores_untrained.append(macro_f1_1)
        micro_f1_2 = f1_score(y_test, y_pred_2, average='micro')
        macro_f1_2 = f1_score(y_test, y_pred_2, average='macro')

        micro_f1_scores_trained.append(micro_f1_2)
        macro_f1_scores_trained.append(macro_f1_2)
        
    return micro_f1_scores_untrained, macro_f1_scores_untrained, micro_f1_scores_trained, macro_f1_scores_trained


In [ ]:
#micro_f1_scores_pnn_untrained1, macro_f1_scores_pnn_untrained1, micro_f1_scores_pnn_trained1, macro_f1_scores_pnn_trained1 = prototypical_scores(range(1,10),train_net)

In [ ]:
micro_f1_scores_pnn_untrained1, macro_f1_scores_pnn_untrained1 = [0.22942113648433352, 0.18335103558151886, 0.2675252257036644, 0.24853956452469464, 0.2738980350504514, 0.30390334572490707, 0.3370950610727562, 0.2870419543281997, 0.32687201274561867] , [0.22054330807850037, 0.1620900578317452, 0.2738878950378947, 0.24662879755051353, 0.2735724840998694, 0.29180368560671016, 0.3332806115442898, 0.27058085506428475, 0.3338911821481306]
micro_f1_scores_pnn_trained1, macro_f1_scores_pnn_trained1= [0.25531067445565586, 0.2295539033457249, 0.2950079660116835, 0.2631439192777483, 0.3021773765268189, 0.32899628252788105, 0.35077004779607013, 0.3272703133297929, 0.3546202867764206] , [0.24177501965673714, 0.22887120834652727, 0.2980791417412594, 0.2581345602835542, 0.30700360507520524, 0.31902800049167496, 0.35147061412912417, 0.3141427684756467, 0.36573836002414395]

In [ ]:
print(micro_f1_scores_pnn_untrained1,",", macro_f1_scores_pnn_untrained1) 
print(micro_f1_scores_pnn_trained1,",", macro_f1_scores_pnn_trained1)

In [ ]:
#micro_f1_scores_pnn_untrained2, macro_f1_scores_pnn_untrained2, micro_f1_scores_pnn_trained2, macro_f1_scores_pnn_trained2 = prototypical_scores(range(1,10),train_net2)

In [ ]:
micro_f1_scores_pnn_untrained2, macro_f1_scores_pnn_untrained2 = [0.22942113648433352, 0.18335103558151886, 0.2675252257036644, 0.24853956452469464, 0.2738980350504514, 0.30390334572490707, 0.3370950610727562, 0.2870419543281997, 0.32687201274561867] , [0.22054330807850037, 0.1620900578317452, 0.2738878950378947, 0.24662879755051353, 0.2735724840998694, 0.29180368560671016, 0.3332806115442898, 0.27058085506428475, 0.3338911821481306]
micro_f1_scores_pnn_trained2, macro_f1_scores_pnn_trained2= [0.2709771640998407, 0.24880509824747743, 0.3133297928836962, 0.27841210833775887, 0.32129580456718004, 0.3510355815188529, 0.363781200212427, 0.35422198619224643, 0.3757302177376527] , [0.25824969166722284, 0.253608824380797, 0.3160829108105455, 0.2731251871634769, 0.33121537313031374, 0.3470117382176801, 0.3681217680038476, 0.34575884750273883, 0.3828433342076799]

In [ ]:
print(micro_f1_scores_pnn_untrained2,",", macro_f1_scores_pnn_untrained2) 
print(micro_f1_scores_pnn_trained2,",", macro_f1_scores_pnn_trained2)

In [ ]:
plt.figure(figsize=(15, 5))
k_values= range(2,11)
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_pnn_untrained1,marker="o",label="Red Prototípica sin entrenamiento")
plt.plot(k_values, micro_f1_scores_pnn_trained1, marker="o", label="Red Prototípica con entrenamiento")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(k_values, macro_f1_scores_pnn_untrained1,marker="o",label="Red Prototípica sin entrenamiento")
plt.plot(k_values, macro_f1_scores_pnn_trained1, marker="o", label="Red Prototípica con entrenamiento")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()

plt.grid(True)
plt.savefig('grafica9.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
k_values= range(2,11)
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(k_values, micro_f1_scores_pnn_untrained2,marker="o",label="Red Prototípica sin entrenamiento")
plt.plot(k_values, micro_f1_scores_pnn_trained1, marker="o", label="Red Prototípica con entrenamiento")
plt.plot(k_values, micro_f1_scores_pnn_trained2, marker="o", label="Red Prototípica con entrenamiento por clases")

plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)

plt.plot(k_values, macro_f1_scores_pnn_untrained2,marker="o",label="Red Prototípica sin entrenamiento")
plt.plot(k_values, macro_f1_scores_pnn_trained1, marker="o", label="Red Prototípica con entrenamiento")
plt.plot(k_values, macro_f1_scores_pnn_trained2, marker="o", label="Red Prototípica con entrenamiento por clases")

plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()

plt.grid(True)
plt.savefig('grafica10.png', bbox_inches='tight')
plt.show()

## Semi Supervisado

### Unsupervised Data Augmentation

In [ ]:
from tqdm import tqdm
import nlpaug.augmenter.word as naw

#Puede dar problemas con los drivers, incluso al ejecutar en cpu
#aug_bert = naw.ContextualWordEmbsAug(model_path='distilbert-base-uncased', action="substitute", device='cpu')

aug_syn = naw.SynonymAug(aug_src='wordnet')

aug_swap = naw.RandomWordAug(action="swap", aug_p=0.1)

def apply_custom_augmentation_example(X, bert_aug=1, syn_aug=1, swap_aug=1):
    #text_res = aug_bert.augment(X)[:bert_aug]
    text_res += aug_syn.augment(X)[:syn_aug]
    text_res += aug_swap.augment(X)[:swap_aug]
    return text_res




In [ ]:
def UDA_scores(k_values,model_function, unsupervised_train_algorithm, net_params=dict(), unsupervised_train_params=dict(),):
    micro_f1_scores = []
    macro_f1_scores = []

    for k in k_values:
        train = train_fs_list[k]

            
        X = train['text'].values
        X_ul = train_ul['text'].values
        X_test = test['text'].values


        y = train['label'].values
        y_val = val['label'].values
        y_test = test['label'].values
        y_test = tf.constant(y_test, dtype=tf.int32)
        
        model = model_function(X,y,**net_params)
        
        unsupervised_train_algorithm(model=model,X=X_train, y=y_train, X_ul=X_ul[:1000], **unsupervised_train_params)
        
        y_pred_proba = model.predict(X_test)
        y_pred = np.argmax(y_pred_proba, axis=1)
        micro_f1 = f1_score(y_test, y_pred, average='micro')
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        print(micro_f1)
        print(macro_f1)
        micro_f1_scores.append(micro_f1)
        macro_f1_scores.append(macro_f1)
        
    return micro_f1_scores, macro_f1_scores


In [ ]:
def train_step_hybrid(model, x_labeled, y_labeled, x_unlabeled, optimizer, temperature, bert_aug, syn_aug, swap_aug, unsup_weight):
    loss_function = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False, 
        reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE
    )
    
    all_augs = []
    repeats = []
    
    for x_ul in x_unlabeled:
        aug_examples = apply_custom_augmentation_example(x_ul, bert_aug=bert_aug, syn_aug=syn_aug, swap_aug=swap_aug)
        all_augs.extend(aug_examples)
        repeats.append(len(aug_examples))
    
    with tf.GradientTape() as tape:
        # --- PERDIDA SUPERVISADA ---
        x_l_input = tf.constant(x_labeled, dtype=tf.string)
        probs_labeled = model(x_l_input, training=True)
        supervised_loss = loss_function(y_labeled, probs_labeled)
        
        # --- PERDIDA DE CONSISTENCIA UDA ---
        x_ul_input = tf.constant(x_unlabeled, dtype=tf.string)
        probs_orig = model(x_ul_input, training=False)
        
        logits_pseudo = tf.math.log(probs_orig + 1e-8)
        probs_orig_sharp = tf.nn.softmax(logits_pseudo / temperature, axis=-1)
        probs_orig_sharp = tf.stop_gradient(probs_orig_sharp) 
        rep_tensor = tf.constant(repeats, dtype=tf.int32)
        probs_orig_sharp_expanded = tf.repeat(probs_orig_sharp, repeats=rep_tensor, axis=0)
        
        x_aug_input = tf.constant(all_augs, dtype=tf.string)
        probs_aug = model(x_aug_input, training=True)
        
        loss_val = tf.keras.losses.categorical_crossentropy(
            probs_orig_sharp_expanded, 
            probs_aug, 
            from_logits=False
        )
        final_consistency_loss = unsup_weight*tf.reduce_mean(loss_val)
        
        total_loss = supervised_loss + final_consistency_loss
        
    grads = tape.gradient(total_loss, model.trainable_variables)
    
    grads_and_vars = [(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None]
    if grads_and_vars:
        gradients, variables = zip(*grads_and_vars)
        clipped_gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        optimizer.apply_gradients(zip(clipped_gradients, variables))
        
    return total_loss, supervised_loss, final_consistency_loss

def train_hybrid(model, X, y, X_ul, batch_size=512, learning_rate=5e-3, num_epochs=20, temperature=0.5, bert_aug=1, syn_aug=1, swap_aug=1, unsup_weight=0.5):
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    dataset_unlabeled = tf.data.Dataset.from_tensor_slices(X_ul).shuffle(2000).batch(batch_size, drop_remainder=False)

    for epoch in range(num_epochs):
        epoch_t_loss, epoch_s_loss, epoch_c_loss = 0.0, 0.0, 0.0
        steps = 0
        
        for batch_x_ul in dataset_unlabeled:
            
            x_ul_batch = [x.numpy().decode('utf-8') for x in batch_x_ul]
            t_loss, s_loss, c_loss = train_step_hybrid(
                model=model,
                x_labeled=X,       
                y_labeled=y,      
                x_unlabeled=x_ul_batch,   
                optimizer=optimizer,
                temperature=temperature,
                bert_aug=bert_aug,
                syn_aug=syn_aug,
                swap_aug=swap_aug,
                unsup_weight=unsup_weight 
            )
            epoch_t_loss += t_loss.numpy()
            epoch_s_loss += s_loss.numpy()
            epoch_c_loss += c_loss.numpy()
            steps += 1
            
        print(f"EPOCH {epoch:02d} | Avg Total Loss: {epoch_t_loss/steps:.6f} | Avg Sup Loss: {epoch_s_loss/steps:.6f} | Avg Unsup Loss: {epoch_c_loss/steps:.6f}")
            
    return model

In [ ]:
UDA_params = best_params_NN.copy()
UDA_params['train'] = False

In [ ]:
#micro_f1_scores_UDA_nn, macro_f1_scores_UDA_nn = UDA_scores(range(1,10,2), create_fs_NN,train_hybrid, net_params=UDA_params)

In [ ]:
micro_f1_scores_UDA_nn, macro_f1_scores_UDA_nn = [0.12626128518321827, 0.1127190653212958, 0.12413701540095592, 0.12413701540095592, 0.12426978226234732] , [0.11474780093700462, 0.0978490064430989, 0.10262992696292195, 0.10760972051866513, 0.10716991240450777]

In [ ]:
print(micro_f1_scores_UDA_nn, ",",macro_f1_scores_UDA_nn)

In [ ]:
plt.figure(figsize=(15, 5))
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(range(2,11), micro_f1_scores_nn, marker="o", label="Entrenamiento Supervisado")
plt.plot(range(2,11,2), micro_f1_scores_UDA_nn, marker="o", label="Entrenamiento UDA")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(range(2,11), macro_f1_scores_nn, marker="o", label="Entrenamiento Supervisado")
plt.plot(range(2,11,2), macro_f1_scores_UDA_nn, marker="o", label="Entrenamiento UDA")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()

plt.grid(True)
plt.savefig('grafica11.png', bbox_inches='tight')
plt.show()

### Uncertainity aware self-training

In [ ]:
def UAST_iter(model_function, X, y, X_ul, percentile_umbral, model_params, aug_params, temperature=0.5):
    model_it = model_function(X, y, **model_params)
    y_ul, un = pseudolabels(model_it, X_ul, aug_params, temperature)
    
    dynamic_umbral = np.percentile(un, percentile_umbral) 
    mask = un <= dynamic_umbral

    if not np.any(mask):
        return X, y, X_ul
    else:
        X_safe = X_ul[mask]
        y_safe = y_ul[mask]
        
        X_new = np.vstack((X, X_safe)) if len(X.shape) > 1 else np.concatenate((X, X_safe))
        y_new = np.concatenate((y, y_safe.flatten()))
        X_ul_remaining = X_ul[~mask]

        return X_new, y_new, X_ul_remaining


def pseudolabels(model_it, X_ul, aug_params, temperature=0.5):
    y_ul = []
    un = []

    for x in X_ul:
        X_aug = apply_custom_augmentation_example(x, **aug_params)
        all_texts = [x] + X_aug 
        x_inputs = tf.constant(all_texts, dtype=tf.string)
        
        probs = model_it.predict(x_inputs, verbose=0) 
        avg_prob = np.mean(probs, axis=0)
        
        p_temp = np.power(avg_prob, 1/temperature)
        sharpened_prob = p_temp / np.sum(p_temp, axis=-1, keepdims=True)
        
        y = np.argmax(sharpened_prob)
        uncertainty = -np.sum(avg_prob * np.log(avg_prob + 1e-10), axis=-1)
        
        y_ul.append(y)
        un.append(uncertainty)

    return np.array(y_ul), np.array(un)

In [ ]:
def UAST(model_function, X, y, X_ul, percentile_umbral=10, model_params=dict(), aug_params=dict(), temperature=0.5):
    for i in range(4):
        print("=======Iteración ", i+1)
        print("Cantidad de ejemplos etiquetados:", X.shape[0])
        print("Cantidad de ejemplos sin etiquetar:", X_ul.shape[0])
        X, y, X_ul = UAST_iter(model_function, X, y, X_ul, percentile_umbral, model_params, aug_params, temperature)
        
    print("=======Final")
    print("Cantidad de ejemplos etiquetados:", X.shape[0])
    print("Cantidad de ejemplos sin etiquetar:", X_ul.shape[0])
    print("Entrenamiento del modelo:")
    
    model = model_function(X, y, **model_params)
    
    return model, X, y, X_ul

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

def UAST_scores(k_values,model_function, uast_params=dict(),model_params=dict(),aug_params=dict()):

    micro_f1_scores = []
    macro_f1_scores = []

    for k in k_values:
        train = train_fs_list[k]

            
        X = train['text'].values
        X_ul = train_ul['text'].values
        X_test = test['text'].values


        y = train['label'].values
        y_test = test['label'].values
        y_test = tf.constant(y_test, dtype=tf.int32)


        model, _,_,_ = UAST(model_function, X, y, X_ul[:1000],model_params=model_params,aug_params=aug_params,**uast_params)
        y_pred_proba = model.predict(X_test)
        y_pred = np.argmax(y_pred_proba, axis=1)
                   
        micro_f1 = f1_score(y_test, y_pred, average='micro')
        macro_f1 = f1_score(y_test, y_pred, average='macro')

        micro_f1_scores.append(micro_f1)
        macro_f1_scores.append(macro_f1)
        
    return micro_f1_scores, macro_f1_scores


In [ ]:
#micro_f1_scores_UAST, macro_f1_scores_UAST = UAST_scores(range(1,10,2),create_fs_NN, model_params=best_params_NN)

In [ ]:
micro_f1_scores_UAST, macro_f1_scores_UAST = [0.07408390865639937, 0.09532660647902283, 0.09174190122145512, 0.11258629845990441, 0.1610462028677642] , [0.05774124832562612, 0.08516991631278167, 0.08191387646521853, 0.10382844875505134, 0.15503759082717497]


In [ ]:
print(micro_f1_scores_UAST, ",",macro_f1_scores_UAST)

In [ ]:
plt.figure(figsize=(15, 5))
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(range(2,11), micro_f1_scores_nn,marker="o",label="Red neuronal sin UAST")
plt.plot(range(2,11,2), micro_f1_scores_UAST, marker="o", label="Red neuronal con UAST")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")
plt.legend()
plt.grid(True)


plt.subplot(1,2,2)
plt.plot(range(2,11), macro_f1_scores_nn,marker="o",label="Red neuronal sin UAST")
plt.plot(range(2,11,2), macro_f1_scores_UAST, marker="o", label="Red neuronal con UAST")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.legend()

plt.grid(True)
plt.savefig('grafica12.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
plt.axis('off')
plt.title("Micro-F1 y Macro-F1 en función de los ejemplos de cada clase (Few-Shot)")
plt.subplot(1,2,1)
plt.plot(range(2,11), micro_f1_scores_nn,marker="o")
plt.plot(range(2,11), micro_f1_scores_DISILBERT,marker="o")
plt.plot(range(2,11), micro_f1_scores_promt_RAG,marker="o")
plt.plot(range(2,11), micro_f1_scores_snn_trained,marker="o")
plt.plot(range(2,11), micro_f1_scores_mnn_trained2,marker="o")
plt.plot(range(2,11), micro_f1_scores_pnn_trained2,marker="o")
plt.plot(range(2,11,2), micro_f1_scores_UDA_nn,marker="o")
plt.plot(range(2,11,2), micro_f1_scores_UAST, marker="o")
plt.xlabel("Cantidad de ejemplos por clase)")
plt.ylabel("Micro F1-score")


plt.subplot(1,2,2)
plt.plot(range(2,11), macro_f1_scores_nn,marker="o")
plt.plot(range(2,11), macro_f1_scores_DISILBERT,marker="o")
plt.plot(range(2,11), macro_f1_scores_promt_RAG,marker="o")
plt.plot(range(2,11), macro_f1_scores_snn_trained,marker="o")
plt.plot(range(2,11), macro_f1_scores_mnn_trained2,marker="o")
plt.plot(range(2,11), macro_f1_scores_pnn_trained2,marker="o")
plt.plot(range(2,11,2), macro_f1_scores_UDA_nn,marker="o")
plt.plot(range(2,11,2), macro_f1_scores_UAST, marker="o")
plt.xlabel("Cantidad de ejemplos por clase")
plt.ylabel("Macro F1-score")
plt.savefig('grafica13.png', bbox_inches='tight')
plt.show()